# Finding RBPs via literature / known phages

**Goal of this notebook:** the host-identification step of the metagenome-RBP pipeline.

```
host genomes (this notebook)            metagenomes (later)
  9-6-2 / 9-2-1 / 10-10-2                  MG08.faa / MG09.faa
        |                                        ^
   1. WHO are they?  ── identify closest         |
        |               NAMED relative on NCBI    |
   2. find their phages on NCBI                   |
        |                                         |
   3. collect those phages' RBPs  ────────────────┘  search for these
```

This notebook covers **steps 1–2 groundwork**: figure out what species our isolates are
(so we can search NCBI for phages that infect them). See `data_overview.md` for the raw
sequencing summary; numbers there are reproduced in `data_overview.ipynb`.

## What does ANI mean?

**ANI = Average Nucleotide Identity** — the average percent identity of all the orthologous
(shared) stretches of DNA between two genomes. You chop both genomes into fragments, align
the matching ones, and average the % identity.

- **ANI ≥ ~95–96 % ⇒ same species.** This is the accepted, widely-validated microbial
  species boundary (it corresponds to ~70 % DNA–DNA hybridization, the old gold standard).
- **ANI 95–99 % ⇒ same species, different strain.** Our isolates vs *Pseudomonas* sp. CM27.
- **ANI ~100 % ⇒ same strain / clone.** (Sites `9-6-2` and `9-2-1` to each other.)
- Two fields the sourmash report gave us:
  - `query_ani_at_rank` — the estimated ANI to the best-matching reference (0.995 → 99.5 %).
  - `fraction` / `f_weighted_at_rank` — how much of our genome is *covered* by that reference
    (0.86 → 86 %); the uncovered part is strain-specific DNA the reference lacks.

So our `query_ani_at_rank ≈ 0.995` is what tells us the isolates are the **same species** as the
reference, while only 86 % coverage tells us they are a **distinct strain**, not that exact genome.

## Where our genomes sit right now

`sourmash` placed all three isolates in GTDB species `Pseudomonas_E sp013179515`. That epithet
is not arbitrary — GTDB builds `sp<digits>` names from the **GenBank assembly accession** of the
cluster's representative genome:

| | |
|---|---|
| GTDB species | `Pseudomonas_E sp013179515` |
| → NCBI assembly | **GCA_013179515.1** (RefSeq GCF_013179515.1) |
| Organism | **Pseudomonas sp. CM27** — strain CM27, **tax_id 2738452** |
| Origin | root of *Crithmum maritimum*, Crete, Greece (Scaffold-level, SPAdes) |

**Consequences:**
- Our exact isolates are **not in NCBI** (newly sequenced, not deposited → no accession to look up).
- Their species *is* in NCBI, but only as an **unnamed** "Pseudomonas sp. CM27". There is no
  validly published Latin species name, so there are almost certainly **no phages catalogued
  against this exact host name.**
- To find candidate phages we therefore need the **closest *named* relative**, not the CM27 tag.

## How to find the closest *named* relative

Three standard routes, weakest→strongest resolution:

| Method | Resolution | Cost | Tooling |
|---|---|---|---|
| **A. 16S rRNA BLAST** vs NCBI | genus → species group (16S is ~99 % identical across many *Pseudomonas* spp., so it rarely pins one species) | seconds–minutes, remote | Biopython `qblast` |
| **B. Whole-genome ANI** vs named type-strain genomes | **species (definitive)** — apply the 95 % rule | download refs + ANI run | `pyfastani` (pip FastANI) |
| **C. Web services** — NCBI ANI report, or **TYGS** (dDDH against all type strains) | species, curated | upload genome, manual | browser |

We run **A** for a quick group placement, then **B** for the definitive species call. Both are
executed below.

In [1]:
import re, glob, hashlib, json
from pathlib import Path
import pandas as pd

GEN = Path("../../data/bacteria_genome")
OUT = Path("out_finding_rbps"); OUT.mkdir(exist_ok=True)

# one representative per distinct strain (9-6-2 ≈ 9-2-1; 10-10-2 is distinct)
REPS = {
    "9-6-2":   GEN/"9-6-2/rerun_QB9Y48/annotation",
    "10-10-2": GEN/"10-10-2/annotation",
}

### Step A.1 — extract the 16S rRNA gene from each genome
Bakta annotates the rRNA operons; we pull the 16S (`rrs`) genes from the `.ffn` and collapse the ~7 identical operon copies to the unique sequence(s).

In [2]:
def read_fasta(p):
    name, seq = None, []
    for line in Path(p).read_text().splitlines():
        if line.startswith(">"):
            if name: yield name, "".join(seq)
            name, seq = line[1:], []
        else:
            seq.append(line.strip())
    if name: yield name, "".join(seq)

records = []
for site, ann in REPS.items():
    ffn = glob.glob(str(ann/"*_reference.ffn"))[0]
    copies = {n.split()[0]: s for n, s in read_fasta(ffn) if "16S ribosomal RNA" in n}
    uniq = {}
    for lt, s in copies.items():
        uniq.setdefault(hashlib.md5(s.encode()).hexdigest(), (lt, s))
    lt, s = next(iter(uniq.values()))           # representative copy
    records.append({"site": site, "n_16S_copies": len(copies),
                    "n_unique": len(uniq), "rep_locus": lt, "length": len(s)})
    (OUT/"rep_16S.fasta")  # ensure dir
pd.DataFrame(records).set_index("site")

,n_16S_copies,n_unique,rep_locus,length
site,,,,
9-6-2,7,3,PICBEH_00120,1537
10-10-2,7,2,BPBBIK_00120,1537


In [3]:
# write the representative 16S sequences to a FASTA for BLAST
with open(OUT/"rep_16S.fasta", "w") as fh:
    for site, ann in REPS.items():
        ffn = glob.glob(str(ann/"*_reference.ffn"))[0]
        copies = {n.split()[0]: s for n, s in read_fasta(ffn) if "16S ribosomal RNA" in n}
        uniq = {}
        for lt, s in copies.items():
            uniq.setdefault(hashlib.md5(s.encode()).hexdigest(), (lt, s))
        lt, s = next(iter(uniq.values()))
        fh.write(f">{site}_{lt}\n{s}\n")
print("wrote", OUT/"rep_16S.fasta")

wrote out_finding_rbps/rep_16S.fasta


### Step A.2 — BLAST the 16S against NCBI

Remote `blastn`/megablast against `nt`, restricted to `Pseudomonas[Organism]` so every hit is
a named *Pseudomonas*. We BLAST **both distinct strains** (`9-6-2` and `10-10-2`; `9-2-1` ≈
`9-6-2`). ~1 min each over the network; results cache to `out_finding_rbps/blast16s_by_site.json`,
so re-runs are instant.

> **Why not the curated type-strain 16S DB?** NCBI's legacy remote BLAST API (what Biopython
> drives) does **not** reliably serve the `16S_ribosomal_RNA` database — it returns 0 hits
> silently. For the *curated* type-strain answer use the **web BLAST UI** (pick "16S ribosomal
> RNA sequences") or a local `blast+` with that DB downloaded. We use `nt`+`Pseudomonas` here.
>
> **Caveat:** within *Pseudomonas*, 16S is highly conserved — expect several species at
> 99–100 % identity. Treat A as "which species *group*", then disambiguate with ANI (Step B).

In [4]:
CACHE = OUT/"blast16s_by_site.json"        # {site: [hits...]}, one BLAST per distinct strain

def blast_16s(fasta=OUT/"rep_16S.fasta", force=False):
    if CACHE.exists() and not force:
        return json.loads(CACHE.read_text())
    from Bio.Blast import NCBIWWW, NCBIXML
    out = {}
    for rec in (r for r in open(fasta).read().split(">") if r.strip()):
        site = rec.splitlines()[0].split("_")[0]      # ">9-6-2_PICBEH_00120" -> "9-6-2"
        seq = "".join(rec.splitlines()[1:])
        # nt restricted to Pseudomonas: the legacy remote API does not serve the
        # curated 16S_ribosomal_RNA DB reliably (returns 0 hits silently).
        h = NCBIWWW.qblast("blastn", "nt", seq, hitlist_size=20, megablast=True,
                           entrez_query="Pseudomonas[Organism]")
        b = NCBIXML.read(h)
        out[site] = [{"title": al.title[:110], "ref_len": al.length,
                      "pct_id": round(100*al.hsps[0].identities/al.hsps[0].align_length, 2),
                      "aln_len": al.hsps[0].align_length, "evalue": al.hsps[0].expect}
                     for al in b.alignments]
    CACHE.write_text(json.dumps(out, indent=1))
    return out

by_site = blast_16s()                       # cached if already run
# top 6 hits per distinct strain
blast_df = pd.concat({s: pd.DataFrame(h).head(6) for s, h in by_site.items()},
                     names=["site", "rank"])
blast_df[["pct_id", "title"]]

pct_id                                              title
site    rank                                                           
9-6-2   0      99.80  gi|2744742676|gb|CP158159.1| Pseudomonas putid...
        1      99.80  gi|2623565045|gb|CP137621.1| Pseudomonas putid...
        2      99.80  gi|2197957769|gb|CP069080.1| Pseudomonas putid...
        3      99.80  gi|2295324123|gb|CP079827.1| Pseudomonas sp. H...
        4      99.80  gi|3062544045|gb|CP199806.1| Pseudomonas putid...
        5      99.80  gi|2475877576|gb|CP120969.1| Pseudomonas putid...
10-10-2 0      99.87  gi|2744742676|gb|CP158159.1| Pseudomonas putid...
        1      99.87  gi|2623565045|gb|CP137621.1| Pseudomonas putid...
        2      99.87  gi|2197957769|gb|CP069080.1| Pseudomonas putid...
        3      99.87  gi|2295324123|gb|CP079827.1| Pseudomonas sp. H...
        4      99.87  gi|3062544045|gb|CP199806.1| Pseudomonas putid...
        5      99.87  gi|2475877576|gb|CP120969.1| Pseudomonas putid...

**Result — both distinct strains land in the _Pseudomonas putida_ group:**

- **`9-6-2`** (≈ `9-2-1`): top hits are *P. putida* strains tied at **99.80 %** 16S identity,
  plus *P. kurunegalensis* and unnamed *Pseudomonas* sp.
- **`10-10-2`**: same picture — *P. putida* (e.g. strain CRA-04) at **99.87 %**.

So the conclusion holds for **both** unique strains, not just one:

- They are firmly in the **_P. putida_ group** and are **not _P. aeruginosa_** — so
  *P. aeruginosa*-centric phage/RBP assumptions don't transfer.
  *(Evidence: the 16S hits themselves — *P. putida* ~99.8 %, nothing near *P. aeruginosa* —
  together with CM27 being the genome's GTDB cluster representative. The GTDB genus label
  `Pseudomonas_E` is the split-out genus that carries the *P. putida* group, consistent with
  this; but GTDB genus letters can shift between releases, so treat it as corroboration, not
  proof.)*
- 16S **cannot** resolve *which* P.-putida-group species (all tie ~99.8 %), and our genome is
  its own GTDB species cluster (CM27) → **run Step B (ANI)** for the definitive nearest *named*
  species before committing to a phage host. Note ANI may not even be on the critical path:
  phage host range often spans species *within* a group, so the group-level call may suffice to
  start collecting candidate phages.

### Step B — confirm the species with whole-genome ANI

16S only resolves the *group*; **ANI resolves the species** (≥ 95 % ANI = same species). We
compute ANI of our 3 genomes against three references with **`pyfastani`** (a pip-installable
FastANI — no external binary or conda needed):

- **CM27** (`GCA_013179515.1`) — the GTDB cluster representative our isolates sketch-matched.
- **_P. putida_ type strain NBRC 14164** (`GCF_000412675.1`) — the definitive *"is it really
  P. putida?"* yardstick (this assembly is NCBI's reference **and** the nomenclatural type).
- **_P. aeruginosa_ PAO1** (`GCF_000006765.1`) — negative control.

The cell downloads the references with `datasets` (skips if already present) and caches the ANI
matrix to `out_finding_rbps/ani/ani_matrix.csv`.

In [5]:
import subprocess

ANI = OUT/"ani"; ANI.mkdir(parents=True, exist_ok=True)
ANI_CSV = ANI/"ani_matrix.csv"
refdir = ANI/"refs_raw/ncbi_dataset/data"
REF_ACC = ["GCA_013179515.1", "GCF_000412675.1", "GCF_000006765.1"]

# our 3 assemblies + the 3 references (label -> genome fasta)
GENOMES = {
    "9-6-2":   GEN/"9-6-2/rerun_QB9Y48/QB9Y48_1_rerun_sample_2_2_1_reference.fasta",
    "9-2-1":   GEN/"9-2-1/XH254M_1_sample_1_1_reference.fasta",
    "10-10-2": GEN/"10-10-2/XH254M_3_sample_3_1_reference.fasta",
    "CM27 (anchor)":     refdir/"GCA_013179515.1/GCA_013179515.1_ASM1317951v1_genomic.fna",
    "P.putida type":     refdir/"GCF_000412675.1/GCF_000412675.1_ASM41267v1_genomic.fna",
    "P.aeruginosa PAO1": refdir/"GCF_000006765.1/GCF_000006765.1_ASM676v1_genomic.fna",
}

def _contigs(p):                                              # reuse read_fasta (no Biopython needed)
    return [s.encode() for _, s in read_fasta(p)]

def compute_ani():
    import pyfastani
    if not refdir.exists():                                  # fetch references once
        subprocess.run(["datasets", "download", "genome", "accession", *REF_ACC,
                        "--include", "genome", "--filename", str(ANI/"refs.zip")], check=True)
        subprocess.run(["unzip", "-o", "-q", str(ANI/"refs.zip"), "-d", str(ANI/"refs_raw")], check=True)
    sk = pyfastani.Sketch()
    for name, p in GENOMES.items():
        sk.add_draft(name, _contigs(p))
    mp = sk.index()
    rows = []
    for qn, p in GENOMES.items():                            # FastANI is directional: query -> refs
        hits = {h.name: round(h.identity, 2) for h in mp.query_draft(_contigs(p))}
        rows.append({"query": qn, **{r: hits.get(r) for r in GENOMES}})
    return pd.DataFrame(rows).set_index("query")

ani = pd.read_csv(ANI_CSV, index_col=0) if ANI_CSV.exists() else compute_ani()
if not ANI_CSV.exists():
    ani.to_csv(ANI_CSV)
ani   # rows = query genome, cols = reference; values = % ANI (blank = below FastANI's ~80 % floor)

,9-6-2,9-2-1,10-10-2,CM27 (anchor),P.putida type,P.aeruginosa PAO1
query,,,,,,
9-6-2,100.00,100.00,99.40,99.23,89.42,80.40
9-2-1,100.00,100.00,99.40,99.23,89.43,80.42
10-10-2,99.40,99.40,100.00,99.24,89.42,80.38
CM27 (anchor),99.41,99.41,99.46,100.00,89.29,80.19
P.putida type,89.18,89.18,89.37,89.22,100.00,80.17
P.aeruginosa PAO1,80.34,80.34,80.29,80.22,80.19,100.00


**Result — they are NOT _P. putida_ proper; they are the (unnamed) CM27 species in the _P. putida_ group:**

| comparison | ANI | meaning |
|---|---|---|
| `9-6-2` ↔ `9-2-1` | **100 %** | identical — confirms `9-6-2` is just a re-sequence of the same isolate |
| `10-10-2` ↔ `9-6-2`/`9-2-1` | **99.4 %** | same species, a distinct strain |
| our genomes ↔ **CM27** | **99.2–99.5 %** | **same species** as *Pseudomonas* sp. CM27 (confirms sourmash) |
| our genomes ↔ **_P. putida_ type strain** | **~89 %** | **below the 95 % cutoff → NOT _P. putida_** |
| our genomes ↔ **_P. aeruginosa_ PAO1** | **~80 %** | clearly a different species (negative control behaves) |

**Conclusion.** All three isolates are a single species (`9-6-2` ≡ `9-2-1`; `10-10-2` is a sister
strain at 99.4 %). That species is the **unnamed _Pseudomonas_ sp. CM27** cluster (≥ 99 % ANI),
which sits **inside the _P. putida_ group but ~89 % ANI from _P. putida_ itself — i.e. a distinct,
currently unnamed species, *not* _P. putida_ sensu stricto.** 16S (Step A) correctly named the
*group*; only ANI revealed they are not the species *P. putida*. So the honest label for our hosts
is **"_Pseudomonas_ sp., _P. putida_ group (≈ sp. CM27)"**.

**Implication for phage hunting (pipeline step 2).** There is no validly-named species — and hence
no curated phage collection — for this exact host. Cast the net at the **_P. putida_ group /
_Pseudomonas_ sp.** level rather than one species name: phage host range frequently spans species
within a group, so group-level candidates (plus phages of CM27's near neighbours) are the right
starting set to pull RBP/tail-fiber proteins from.

## Log
- 2026-06-23 — Created. Step 1 host ID: explained ANI; mapped sourmash tag → NCBI GCA_013179515.1 = *Pseudomonas* sp. CM27 (tax_id 2738452); extracted representative 16S per distinct strain; ran NCBI 16S BLAST (Step A) → *P. putida* group for both strains.
- 2026-06-23 — Step B (ANI, `pyfastani`): our genomes are ~89 % ANI to the *P. putida* type strain (**not** *P. putida*), 99.2–99.5 % to CM27 (same species), ~80 % to *P. aeruginosa*. Host label = "*Pseudomonas* sp., *P. putida* group (≈ sp. CM27)". `9-6-2` ≡ `9-2-1` (100 %).